# Chapter 7: Web Crawling Models

* One of the best things you can do when deciding which data to collect is often to ignore the websites altogether. You don’t start a project that’s designed to be large and scalable by looking at a single website and saying, “What exists?” but by saying, “What do I need?” and then finding ways to seek the information that you need from there.

## Dealing with Different Website Layouts

* **HOW DOES GOOGLE (it's straightforward for AI but bot/program?) ABLE TO DO THAT?**
    * One of the most impressive feats of a search engine such as Google is that it manages to extract relevant and useful data from a variety of websites, having no up front knowledge about the website structure itself. Although we, as humans, are able to immediately identify the title and main content of a page (barring instances of extremely poor web design), it is far more difficult to get a bot to do the same thing.
*

In [1]:
import re

import requests
from bs4 import BeautifulSoup

In [2]:
headers = {"User-Agent": "Mr.Robot-Linux_Machine"}

class Content:
    """Common base class for all articles/pages"""

    def __init__(self, url, title, body, topic=None):
        self.url = url
        self.title = title
        self.body = body
        self.topic = topic

    def print(self):
        """Flexible printing function controls output"""
        if self.topic:
            print(f"New article found for topic: {self.topic}")
        print(f"URL: {self.url}")
        print(f"TITLE: {self.title}")
        print(f"BODY:\n{self.body}")


def scrape_CNN(url):
    bs = BeautifulSoup(
        requests.get(url, headers=headers).text,
        "lxml"
    )
    title = bs.find("h1").text
    body = bs.find("div", {"class": "article__content"}).text
    print("\tTITLE:")
    print(title)
    return Content(url, title, body)


def scrape_Brookings(url):
    bs = BeautifulSoup(
        requests.get(url, headers=headers).text,
        "lxml"
    )
    title = bs.find("h1").text
    body = bs.find("div", {"class": "article-content"}).text
    return Content(url, title, body)

In [3]:
url = "https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html"
content = scrape_CNN(url)
content.print()

	TITLE:

      Dogecoin jumps after Elon Musk replaces Twitter bird with Shiba Inu
    
URL: https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html
TITLE: 
      Dogecoin jumps after Elon Musk replaces Twitter bird with Shiba Inu
    
BODY:




New York — 


            Twitter’s traditional bird icon was booted and replaced with an image of a Shiba Inu, an apparent nod to dogecoin, the joke cryptocurrency that CEO Elon Musk is being sued over. 
    

            Musk addressed the change Monday afternoon, tweeting, “as promised” above an image of a year-old conversation in which another user suggested that Musk “just buy Twitter” and “change the bird logo to a doge.” 
    




      Related article
    













CNN/Adobe Stock







Elon Musk's Twitter promised a purge of blue check marks. Instead he singled out one account






            The doge logo appeared on the site two days after Musk asked a judge to throw out a $258 billion racketeering lawsu

In [4]:
url = "https://www.brookings.edu/research/robotic-rulemaking/"
content = scrape_Brookings(url)
content.print()

URL: https://www.brookings.edu/research/robotic-rulemaking/
TITLE: 
            Robotic rulemaking
          
BODY:




Sections























Toggle section navigation

Sections












Print









Series on Regulatory Process and Perspective







Read more from


        Series on Regulatory Process and Perspective
		      









Follow the authors



Bridget C. E. Dooling









Mark Febrizio









See More







      More On
    

Technology & Information
U.S. Economy




								Sub-Topics
								

Regulatory Policy 






Program

Economic Studies 


Center

Center on Regulation and Markets 





As it has rocketed to some 100 million active users in record time, ChatGPT is provoking conversations about the role of artificial intelligence (AI) in drafting written materials such as student exams, news articles, legal pleadings, poems, and more. The chatbot, developed by OpenAI, relies on a large language model (LLM) to respond to user-submitted request

In [5]:
class Website:
    """Contains information aboute website structure"""

    def __init__(self, name, url, title_tag, body_tag):
        self.name = name
        self.url = url
        self.title_tag = title_tag
        self.body_tag = body_tag


class Crawler:

    def get_page(url):
        try:
            html = requests.get(url, headers=headers).text
        except Exception as e:
            print("ERROR:", e)
            return None
        return BeautifulSoup(html, "lxml")

    def safe_get(bs, selector):
        """
        Utility function used to get a content string from a BeautifulSoup
        object and a selector. Returns an empty string if no object is found
        for the given selector.
        """
        selected_elems = bs.select(selector)
        if selected_elems:
            return "\n".join([elem.get_text() for elem in selected_elems])
        return ""

    def get_content(website, path):
        """Extract content from a given page URL"""
        url = website.url+path
        bs = Crawler.get_page(url)
        if bs is not None:
            title = Crawler.safe_get(bs, website.title_tag)
            body = Crawler.safe_get(bs, website.body_tag)
            return Content(url, title, body)
        return Content(url, "", "")

In [6]:
site_data = [
    ['O\'Reilly', 'https://www.oreilly.com', 'h1', 'div.title-description'],
    ['Reuters', 'https://www.reuters.com', 'h1', 'div.ArticleBodyWrapper'],
    ['Brookings', 'https://www.brookings.edu', 'h1', 'div.post-body'],
    ['CNN', 'https://www.cnn.com', 'h1', 'div.article__content']
]
websites = []
for name, url, title, body in site_data:
    websites.append(Website(name, url, title, body))

Crawler.get_content(
    websites[0],
    '/library/view/web-scraping-with/9781491910283'
).print()

Crawler.get_content(
    websites[1],
    '/article/us-usa-epa-pruitt-idUSKBN19W2D0'
).print()

Crawler.get_content(
    websites[2],
    '/blog/techtank/2016/03/01/idea-to-retire-old-methods-of-policy-education/'
).print()

Crawler.get_content(
    websites[3],
    '/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html'
).print()

URL: https://www.oreilly.com/library/view/web-scraping-with/9781491910283
TITLE: Web Scraping with Python
BODY:

URL: https://www.reuters.com/article/us-usa-epa-pruitt-idUSKBN19W2D0
TITLE: 
BODY:

URL: https://www.brookings.edu/blog/techtank/2016/03/01/idea-to-retire-old-methods-of-policy-education/
TITLE: Idea to Retire: Old methods of policy education
BODY:

URL: https://www.cnn.com/2023/04/03/investing/dogecoin-elon-musk-twitter/index.html
TITLE: 
      Dogecoin jumps after Elon Musk replaces Twitter bird with Shiba Inu
    
BODY:




New York — 


            Twitter’s traditional bird icon was booted and replaced with an image of a Shiba Inu, an apparent nod to dogecoin, the joke cryptocurrency that CEO Elon Musk is being sued over. 
    

            Musk addressed the change Monday afternoon, tweeting, “as promised” above an image of a year-old conversation in which another user suggested that Musk “just buy Twitter” and “change the bird logo to a doge.” 
    




      Related 

## Crawling Sites Through Search

In [16]:
class Website:
    """Contains information aboute website structure"""

    def __init__(self, name, url, search_url, result_listing, result_url,
                 absolute_url, title_tag, body_tag):
        self.name = name
        self.url = url
        self.search_url = search_url
        self.result_listing = result_listing
        self.result_url = result_url
        self.absolute_url = absolute_url
        self.title_tag = title_tag
        self.body_tag = body_tag

In [17]:
class Crawler:

    def __init__(self, website):
        self.site: Website = website
        self.found = {}
        self.visited = {}

    def get_page(url):
        try:
            html = requests.get(url, headers=headers).text
        except Exception as e:
            print("EXCEPTION:", e)
            return None
        return BeautifulSoup(html, "lxml")

    def safe_get(bs, selector):
        """
        Utility function used to get a content string from a BeautifulSoup
        object and a selector. Returns an empty string if no object is found
        for the given selector.
        """
        selected_elems = bs.select(selector)
        if selected_elems:
            return "\n".join([elem.get_text() for elem in selected_elems])
        return ""

    def get_content(self, url, topic=None):
        """Extract content from a given page URL"""
        bs = Crawler.get_page(url)
        if bs is not None:
            title = Crawler.safe_get(bs, self.site.title_tag)
            body = Crawler.safe_get(bs, self.site.body_tag)
            return Content(url, title, body, topic)
        return Content(url, "", "", topic)

    def search(self, topic):
        """
        Searches a given website for a given topic and records all pages found
        """
        bs = Crawler.get_page(self.site.search_url + topic)
        search_results = bs.select(self.site.result_listing)
        for result in search_results:
            url = result.select(self.site.result_url)[0].attrs["href"]
            # Check to see whether it's a relative or an absolute URL
            url = url if self.site.absolute_url else self.site.url + url
            if url not in self.found:
                self.found[url] = self.get_content(url, topic)
            self.found[url].print()

    def crawl(self):
        """Get pages from website home page."""
        bs = Crawler.get_page(self.site.url)
        target_pages = bs.find_all("a", href=re.compile(self.site.target_pattern))
        for target_page in target_pages:
            url = target_page.attrs["href"]
            url = url if self.site.absolute_url else f"{self.site.url}{target_page}"
            if url not in self.visited:
                self.visited[url] = self.get_content(url)
                self.visited[url].print()

In [18]:
site_data = [
    [
        "Reuters",
        "https://reuters.com",
        "https://reuters.com/search/news?blob=",
        "div.search-result-indiv",
        "h3.search-result-title a",
        False,
        "h1",
        "div.ArticleBodyWrapper"
    ],
    [
        "Brookings",
        "https://brookings.edu",
        "https://brookings.edu/search/?s",
        "div.article-info",
        "h4.title a",
        True,
        "h1",
        "div.core-block"
    ],
]

sites = []
for name, url, search, rlisting, rurl, abs_url, tt, bt in site_data:
    sites.append(Website(name, url, search, rlisting, rurl, abs_url, tt, bt))

crawlers = [Crawler(site) for site in sites]
topics = ["python", "data%20science"]

for topic in topics:
    for crawler in crawlers:
        crawler.search(topic)

## Crawling Sites Through Links

In [19]:
class Website:
    def __init__(
        self, name, url, target_pattern, absolute_url, title_tag, body_tag
    ):
        self.name = name
        self.url = url
        self.target_pattern = target_pattern
        self.absolute_url = absolute_url
        self.title_tag = title_tag
        self.body_tag = body_tag

In [20]:
brookings = Website(
    "Brookings",
    "https://brookings.edu",
    "/(research|blog)/",
    True,
    "h1",
    "div.post-body"
)
crawler = Crawler(brookings)
crawler.crawl()

## Crawling Multiple Page Types

In [21]:
class Website:
    def __init__(self, name, url, title_tag, body_tag, page_type):
        self.name = name
        self.url = url
        self.title_tag = title_tag
        self.body_tag = body_tag
        self.page_type = page_type


class Product(Website):
    """Contains information for scraping a product page"""
    def __init__(self, name, url, title_tag, product_number_tag, price_tag):
        Website.__init__(self, name, url, title_tag)
        self.product_number_tag = product_number_tag
        self.price_tag = price_tag

class Article(Website):
    """Contains information for scraping an article page"""
    def __init__(self, name, url, title_tag, body_tag, date_tag):
        Website.__init__(self, name, url, title_tag)
        self.body_tag = body_tag
        self.date_tag = date_tag